In [5]:
import torch

# The following is vibe-coded with Gemini; fixed by Kevin Lim and Zihan Guo

# Use float64 for maximum precision with exponential terms
torch.set_default_dtype(torch.float64)

def calculate_J(R1, R2, S0, theta):

    # Slicing the parameters
    idx = 0
    nus1 = theta[idx : idx + R1]; idx += R1
    sigs1_raw = theta[idx : idx + R1]; idx += R1
    nus2 = theta[idx : idx + R2]; idx += R2
    sigs2_raw = theta[idx : idx + R2]; idx += R2

    nus1 = torch.cat([nus1, torch.tensor([S0])])
    nus2 = torch.cat([torch.tensor([S0]), nus2])

    # Enforce strict positivity on sigmas using softplus
    sigs1 = torch.nn.functional.softplus(sigs1_raw) + 1e-6
    sigs2 = torch.nn.functional.softplus(sigs2_raw) + 1e-6

    # Calculate Unit Base Coefficients (Setting lam1 = 1, lam2 = 1)

    # Left Side
    c_v1_unit = []
    dist0 = nus1[1] - nus1[0]
    c_v1_unit.append((-torch.exp(-dist0/sigs1[0]), torch.exp(dist0/sigs1[0])))

    # recursively calculating c^1
    for j in range(R1 - 1):
        P = c_v1_unit[j][0] + c_v1_unit[j][1]
        S = (1/sigs1[j]) * (-c_v1_unit[j][0] + c_v1_unit[j][1])
        dist = nus1[j+2] - nus1[j+1]
        c_v1_unit.append((0.5 * (P - sigs1[j+1]*S) * torch.exp(-dist/sigs1[j+1]),
                          0.5 * (P + sigs1[j+1]*S) * torch.exp(dist/sigs1[j+1])))

    # Right Side
    c_v2_unit = [None] * R2
    dist_u = nus2[-1] - nus2[-2]
    c_v2_unit[-1] = ((torch.exp(dist_u/sigs2[-1]), -torch.exp(-dist_u/sigs2[-1])))

    # recursively calculating c^2
    for j in range(R2 - 1, 0, -1):
        P = c_v2_unit[j][0] + c_v2_unit[j][1]
        S = (1/sigs2[j]) * (-c_v2_unit[j][0] + c_v2_unit[j][1])
        dist = nus2[j] - nus2[j-1]
        c_v2_unit[j-1] = (0.5 * (P - sigs2[j-1]*S) * torch.exp(dist/sigs2[j-1]),
                          0.5 * (P + sigs2[j-1]*S) * torch.exp(-dist/sigs2[j-1]))

    # Unscaled left-side value at S0
    v1 = c_v1_unit[-1][0] + c_v1_unit[-1][1]

    # Unscaled right-side value at S0
    v2 = c_v2_unit[0][0] + c_v2_unit[0][1]

    # Unscaled left-side derivative at S0
    Dk_v1 = (1/sigs1[-1]) * (-c_v1_unit[-1][0] + c_v1_unit[-1][1])

    # Unscaled right-side derivative at S0
    Dk_v2 = (1/sigs2[0]) * (-c_v2_unit[0][0] + c_v2_unit[0][1])

    # Values for lambda1 and lambda2
    lam1 = v2 / (Dk_v1 * v2 - v1 * Dk_v2)
    lam2 = v1 / (Dk_v1 * v2 - v1 * Dk_v2)

    c_v1 = [(c1 * lam1, c2 * lam1) for (c1, c2) in c_v1_unit]
    c_v2 = [(c1 * lam2, c2 * lam2) for (c1, c2) in c_v2_unit]

    jumps = []

    # Left Side Jumps
    for j in range(R1 - 1):
        v_left = (1/sigs1[j]**2) * (c_v1[j][0] + c_v1[j][1])
        dist = nus1[j+2] - nus1[j+1]
        v_right = (1/sigs1[j+1]**2) * (c_v1[j+1][0]*torch.exp(dist/sigs1[j+1]) +
                                             c_v1[j+1][1]*torch.exp(-dist/sigs1[j+1]))
        jumps.append(v_right - v_left)

    # Jump at S0
    v1_S0 = (1/sigs1[-1]**2) * (c_v1[-1][0] + c_v1[-1][1])
    v2_S0 = (1/sigs2[0]**2) * (c_v2[0][0] + c_v2[0][1])

    jumps.append(v2_S0 - v1_S0)

    # Right Side Jumps
    for j in range(R2 - 1):
        v_right = (1/sigs2[j+1]**2) * (c_v2[j+1][0] + c_v2[j+1][1])
        dist = nus2[j+1] - nus2[j]
        v_left = (1/sigs2[j]**2) * (c_v2[j][0]*torch.exp(-dist/sigs2[j]) +
                                             c_v2[j][1]*torch.exp(dist/sigs2[j]))
        jumps.append(v_right - v_left)

    j = torch.sum(torch.stack(jumps) ** 2)
    return j, lam1, lam2

In [28]:
R1, R2 = 4, 3
S0 = 1271.87

initial_sigs1 = torch.tensor([149.0, 130.0, 250.0, 160.0])
initial_sigs2 = torch.tensor([155.0, 180.0, 210.0])
sigs1_raw = torch.log(torch.exp(initial_sigs1) - 1)
sigs2_raw = torch.log(torch.exp(initial_sigs2) - 1)

theta_old = torch.cat([
  torch.linspace(800, 1200, R1),   # nus1
  sigs1_raw,                       # sigs1_raw
  torch.linspace(1400, 2200, R2),  # nus2
  sigs2_raw                        # sigs2_raw
]).clone().detach().requires_grad_(True)

theta_initial = theta_old.clone().detach()

J_old, lam1, lam2 = calculate_J(
  R1, R2, S0, theta_old
)
J_old.backward()
old_gradients = theta_old.grad.clone().detach()

print("=== Current Values ===")
print(f"Current Smoothness J_old: {J_old.item():.12f}")
print(f"Calculated Lambda 1 (Left Wing Scaling):  {lam1.item():.12f}")
print(f"Calculated Lambda 2 (Right Wing Scaling): {lam2.item():.12f}\n")

# initial test with discrete alphas
alphas = [10.0, 1.0, 0.1, 0.01, 0.001]
better_theta_found = False
theta_new_values = None
alpha_used = None

for alpha in alphas:
  with torch.no_grad():
  # not to track the operations performed inside this block for gradient calculation
      candidate_theta = theta_old - alpha * old_gradients
      J_candidate, _, _ = calculate_J(
          R1, R2, S0, candidate_theta
      )

      if J_candidate.item() < J_old.item():
          theta_new_values = candidate_theta.clone().detach()
          alpha_used = alpha
          better_theta_found = True
          break

if better_theta_found:
  theta_new = theta_new_values.clone().detach().requires_grad_(True)

  J_new, _, _ = calculate_J(
      R1, R2, S0, theta_new
  )
  J_new.backward()
  new_gradients = theta_new.grad.clone().detach()

  print(f"=== Step Successful (α = {alpha_used}) ===")
  print(f"Improved Smoothness J_new: {J_new.item():.12f}")
  print(f"Total Penalty Reduction: {((J_old - J_new)/J_old * 100).item():.12f}%\n")

  labels = []
  for i in range(R1): labels.append(f"nu1_{i}")
  for i in range(R1): labels.append(f"sig1_raw_{i}")
  for i in range(R2): labels.append(f"nu2_{i}")
  for i in range(R2): labels.append(f"sig2_raw_{i}")

  print(f"{'Parameter':<12} | {'Old Value':<12} | {'Old Gradient':<15} | {'New Value':<12} | {'New Gradient':<15}")
  print("-" * 76)

  for idx in range(len(labels)):
      print(f"{labels[idx]:<12} | "
            f"{theta_old[idx].item():<12.4f} | "
            f"{old_gradients[idx].item():<15.4e} | "
            f"{theta_new[idx].item():<12.4f} | "
            f"{new_gradients[idx].item():<15.4e}")

else:
  print("Could not find an immediate improvement using basic gradient steps.")

=== Current Values ===
Current Smoothness J_old: 0.000003114399
Calculated Lambda 1 (Left Wing Scaling):  4.033944160692
Calculated Lambda 2 (Right Wing Scaling): 0.672975789580

=== Step Successful (α = 10.0) ===
Improved Smoothness J_new: 0.000003114399
Total Penalty Reduction: 0.000001712523%

Parameter    | Old Value    | Old Gradient    | New Value    | New Gradient   
----------------------------------------------------------------------------
nu1_0        | 800.0000     | -1.2335e-09     | 800.0000     | -1.2335e-09    
nu1_1        | 933.3333     | 6.5566e-10      | 933.3333     | 6.5566e-10     
nu1_2        | 1066.6667    | 7.6856e-09      | 1066.6667    | 7.6856e-09     
nu1_3        | 1200.0000    | 2.9298e-08      | 1200.0000    | 2.9298e-08     
sig1_raw_0   | 149.0000     | 1.8462e-09      | 149.0000     | 1.8462e-09     
sig1_raw_1   | 130.0000     | -3.8749e-08     | 130.0000     | -3.8749e-08    
sig1_raw_2   | 250.0000     | 3.2318e-08      | 250.0000     | 3.2318e-0

In [30]:
# class torch.optim.LBFGS(params, lr=1, max_iter=20, max_eval=None, tolerance_grad=1e-07, tolerance_change=1e-09, history_size=100, line_search_fn=None)
# https://docs.pytorch.org/docs/2.12/generated/torch.optim.LBFGS.html#torch.optim.LBFGS.step$0

# line search
line_search = torch.optim.LBFGS([theta_old], max_iter=50, tolerance_grad=-1e-15, tolerance_change=-1e-15, line_search_fn="strong_wolfe")

def closure():
  line_search.zero_grad()
  J, _, _ = calculate_J(R1, R2, S0, theta_old)
  scaled_J = J * 1e7
  scaled_J.backward()
  return scaled_J

line_search.step(closure)

J_optimized, _, _ = calculate_J(R1, R2, S0, theta_old)

# The following print statements are vibe-coded
print(f"Optimized J Value: {J_optimized.item():.12f}\n")

labels = []
labels.extend([f"nu1_{i}" for i in range(R1)])
labels.extend([f"sig1_raw_{i}" for i in range(R1)])
labels.extend([f"nu2_{i}" for i in range(R2)])
labels.extend([f"sig2_raw_{i}" for i in range(R2)])

print(f"{'Parameter':<15} | {'Old Value':<15} | {'Optimized Value':<15}")
print("-" * 53)

for i in range(len(theta_old)):
    print(f"{labels[i]:<15} | "
          f"{theta_initial[i].item():<15.6f} | "
          f"{theta_old[i].item():<15.6f}")

print(f"Total Penalty Reduction: {((J_old - J_optimized)/J_old * 100).item():.12f}%\n")


Optimized J Value: 0.000000000000

Parameter       | Old Value       | Optimized Value
-----------------------------------------------------
nu1_0           | 800.000000      | 804.015937     
nu1_1           | 933.333333      | 927.168334     
nu1_2           | 1066.666667     | 1056.015443    
nu1_3           | 1200.000000     | 1172.581942    
sig1_raw_0      | 149.000000      | 184.602876     
sig1_raw_1      | 130.000000      | 184.602876     
sig1_raw_2      | 250.000000      | 184.602876     
sig1_raw_3      | 160.000000      | 184.602876     
nu2_0           | 1400.000000     | 1406.797316    
nu2_1           | 1800.000000     | 1803.707728    
nu2_2           | 2200.000000     | 2199.869664    
sig2_raw_0      | 155.000000      | 184.602876     
sig2_raw_1      | 180.000000      | 184.602876     
sig2_raw_2      | 210.000000      | 184.602876     
Total Penalty Reduction: 100.000000000000%



## Overall Logic

Given the function for C_θ(K) at the end of page 19 in the LVG paper (z = 1), I simply need to calculate the sum of the squared values of jumps at each ν_j for C_θ''(ν_j). To do this, I first calculate the two c_j^1 and c_j^2 coefficients for the left and right parts, respectively, recursively with two for loops based on the formula specified on page 18 and 19. Then, I take the second derivative of C_θ(K), which is the same as taking the second derivative of V_θ(K). V_θ''(K) = (1/σ_j)^2 V_θ(K), and I can use this to calculate the jumps for C_θ''(ν_j), which thus leads to J(θ).

Now that I have J(θ), I can simply use PyTorch's Autograd to automatically compute the gradient of J(θ) with respect to every component in the vector θ.

## Stage 1:

### Calculate initial values c_1^{1, 1}, c_1^{2, 1}, c_2^{1, 1}, c_2^{2, 1}

To prevent floating-point overflow, use distance from L and from U: `dist0 = nus1[0] - L, dist_u = U - nus2[-1]` (rescaling to prevent the calculated number to get too large to cause overflow)

$$e^{-K/\sigma} = e^{-\nu/\sigma} e^{-(K-\nu)/\sigma}$$

$$c_1^{1, 1} e^{-K/\sigma_1} = (c_1^{1, 1} e^{-\nu_0/\sigma_1}) e^{-(K-\nu_0)/\sigma_1} = (-\lambda_1 e^{L/\sigma_1} e^{-\nu_0/\sigma_1}) e^{-(K-\nu_0)/\sigma_1}$$

We use the following equations to compute the "distance" c^1 and c^2, then normalized by 0.5 * s0 (to map the coefficient space to the slope-matching derivative space required by the spline).

$$c_{1, \text{dist}}^{1, 1} = c_1^{1, 1} e^{-\nu_0/\sigma_1} = c_1^{1, 1} e^{-\nu_0/\sigma_1} = -\lambda_1 e^{L/\sigma_1} e^{-\nu_0/\sigma_1} = -\lambda_1 e^{-\text{dist0}/\sigma_1}$$

(similar for c_1^{2, 1})

Similarly, we have

$$c_{R+1, \text{dist}}^{1, 2} = c_{R+1}^{1, 1} e^{-\nu_R/\sigma_{R+1}} = c_{R+1}^{1, 1} e^{-\nu_R/\sigma_{R+1}} = \lambda_2 e^{U/\sigma_{R+1}} e^{-\nu_R/\sigma_{R+1}} = -\lambda_2 e^{\text{dist_u}/\sigma_{R+1}}$$

(similar for c_2^{2, 2})

Then, we use these initial parameters to calculate c^1 and c^2 to calculate for each j.

Similarly, in the recursive calculations, we use the newly calculated "distance" c^1 and c^2.

In general,

$$c_{j, \text{dist}}^{1, 1} = c_j^{1, 1} e^{-\nu_{j-1}/\sigma_j} = -\lambda_1 e^{-(L-\nu_{j-1})/\sigma_j}$$

$$c_{j, \text{dist}}^{2, 1} = c_j^{2, 1} e^{-\nu_{j-1}/\sigma_j} = \lambda_1 e^{-(L-\nu_{j-1})/\sigma_j}$$

$$c_{j, \text{dist}}^{1, 2} = c_j^{1, 2} e^{-\nu_{j-1}/\sigma_j} = -\lambda_2 e^{-(U-\nu_{j-1})/\sigma_j}$$

$$c_{j, \text{dist}}^{2, 2} = c_j^{1, 2} e^{-\nu_{j-1}/\sigma_j} = \lambda_2 e^{-(U-\nu_{j-1})/\sigma_j}$$

$$V(K) = V^1(K) + V^2(K) = \sum_{j=1}^{R+1}[(c_j^{1, 1} e^{-K/\sigma_j} + c_j^{2, 1} e^{K/\sigma_j}) + (c_j^{1, 2} e^{-K/\sigma_j} + c_j^{2, 2} e^{K/\sigma_j})] \textbf{1}_{[\nu_{j-1}, \nu_j)} (K) = \sum_{j=1}^{R+1} [(c_{j, \text{dist}}^{1, 1} e^{-(K-\nu_{j-1})/\sigma_j} + c_{j, \text{dist}}^{2, 1} e^{(K-\nu_{j-1})/\sigma_j}) + (c_{j, \text{dist}}^{1, 2} e^{-(K-\nu_{j-1})/\sigma_j} + c_{j, \text{dist}}^{2, 2} e^{(K-\nu_{j-1})/\sigma_j})] \textbf{1}_{[\nu_{j-1}, \nu_j)} (K)$$

where $$K - \nu_j = \text{distance}$$

### Recursively calculating "distance" c_j^{1, 1}, c_j^{2, 1}, c_j^{1, 1}, c_j^{2, 1}

$$P = c_{j, \text{dist}}^{1, 1} + c_{j, \text{dist}}^{2, 1}$$
$$S = \frac{1}{\sigma_j} (-c_{j, \text{dist}}^{1, 1} + c_{j, \text{dist}}^{2, 1})$$
$$c_{j+1, \text{dist}}^{1, 1} = 0.5(P - \sigma_{j+1} S)$$
$$c_{j+1, \text{dist}}^{2, 1} = 0.5(P + \sigma_{j+1} S)$$

## Stage 2:

Solve

$$\lambda_1 V^1_{unit}(K) = \lambda_2 V^2_{unit}(K)$$
$$\lambda_1 \partial_K V^1_{unit}(K) - \lambda_2 \partial_K V^2_{unit}(K) = 1$$

to obtain lam1 and lam2.

## Stage 3:

### Scale c^1, c^2 calculated in Stage 1 by lam1, lam2

Because we used initial values of lam1 = 1, lam2 = 1 to calculate the initial "unit" c^1, c^2, we need to scale them by lam1 and lam2 we solved in Stage 2 to obtain the actual c^1, c^2 (as indicated by the formulas for the initial values of c^1, c^2) so that we can use the actual values to calculate smoothness.

## Stage 4

Using `j_score.backward()` and `theta.grad` to calculate the gradients of J with respect to each component in theta.